# Routing experiment on a free Colab T4

Runs the minimal first experiment (see `docs/research-report.md`): a small and a large Qwen model
answer HotpotQA questions, then routers learn when to escalate from small to large.

1. **Runtime → Change runtime type → T4 GPU.**
2. **Runtime → Run all.**

Data and results are saved to Google Drive in chunks of 500 questions. If the session disconnects,
reconnect and **Run all** again: finished chunks are skipped.

Step 6 runs a 500-question pilot first. Check its answers and time estimate before step 7.

## 1. Settings

In [ ]:
REPO = "https://github.com/mohameddhameem/Custom-LLM-Router.git"
BRANCH = "master"
CONFIG = "configs/t4-vllm.toml"  # switch to "configs/t4-hf.toml" if vLLM fails in step 5
ROUTER_SIZE, CALIB_SIZE = 10_000, 1_000  # train questions to label; the test set is all 7,405 validation questions
DRIVE = "/content/drive/MyDrive/custom-llm-router"
RUN = f"{DRIVE}/runs/t4"

## 2. GPU and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount("/content/drive")

import os
DATA = f"{DRIVE}/data/hotpotqa"
os.makedirs(DATA, exist_ok=True)
os.environ.update(DATA=DATA, RUN=RUN, CONFIG=CONFIG, BIN="/content/venv/bin",
                  ROUTER_SIZE=str(ROUTER_SIZE), CALIB_SIZE=str(CALIB_SIZE))

## 3. Code

If the repository is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar).

In [ ]:
from google.colab import userdata
try:
    url = REPO.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    url = REPO
%cd /content
if not os.path.exists("Custom-LLM-Router"):
    !git clone -q -b {BRANCH} {url}
%cd /content/Custom-LLM-Router
!git pull -q && git log --oneline -1

## 4. Install

A separate virtual environment with Python 3.12, so Colab's own packages are untouched.
Takes a few minutes the first time in each session.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 /content/venv
!uv pip install -q --python /content/venv -e ".[llm]" bitsandbytes
if "vllm" in CONFIG:
    !uv pip install -q --python /content/venv vllm
!/content/venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

## 5. Data

Downloads HotpotQA once to Drive (about 700 MB) and makes the question splits.

In [ ]:
%%bash
set -e
[ -f "$DATA/distractor_validation.parquet" ] || $BIN/prepare-hotpotqa --out "$DATA"
[ -f "$DATA/splits.json" ] || $BIN/make-splits --train "$DATA/distractor_train.parquet" \
    --router-size $ROUTER_SIZE --calib-size $CALIB_SIZE --out "$DATA/splits.json"
ls -lh "$DATA"

## 6. Pilot: 500 questions

Checks that the models load and answer sensibly, and measures speed. If the `small` or `large`
stage fails to start with vLLM, set `CONFIG = "configs/t4-hf.toml"` in step 1, delete
`runs/t4-pilot` on Drive, and run all again.

In [ ]:
%%bash
set -e
for stage in evidence small large merge; do
  $BIN/run-experts --config $CONFIG --data "$DATA/distractor_train.parquet" --splits "$DATA/splits.json" \
      --split router_train --limit 500 --run "$RUN-pilot" --name pilot --stage $stage
done

In [ ]:
%%bash
$BIN/python - <<'EOF'
import os, pandas as pd
df = pd.read_parquet(os.environ["RUN"] + "-pilot/pilot.parquet")
print(df[["answer", "small_pred", "large_pred"]].head(15).to_string())
n = int(os.environ["ROUTER_SIZE"]) + int(os.environ["CALIB_SIZE"]) + 7405
hours = n * (df["small_seconds"].mean() + df["large_seconds"].mean()) / 3600
print(f"\nsmall F1 {df.small_f1.mean():.3f} | large F1 {df.large_f1.mean():.3f}")
print(f"estimated GPU time for all {n:,} questions: {hours:.1f} h")
EOF

## 7. Full run

Cheap stages first, so an interruption during the large model leaves everything else done.
Rerun this cell (or Run all) after a disconnect to continue.

In [ ]:
%%bash
set -e
for stage in evidence small large merge; do
  $BIN/run-experts --config $CONFIG --data "$DATA/distractor_train.parquet" --splits "$DATA/splits.json" \
      --split router_train --run "$RUN" --name router_train --stage $stage
  $BIN/run-experts --config $CONFIG --data "$DATA/distractor_train.parquet" --splits "$DATA/splits.json" \
      --split calib --run "$RUN" --name calib --stage $stage
  $BIN/run-experts --config $CONFIG --data "$DATA/distractor_validation.parquet" \
      --run "$RUN" --name test --stage $stage
done

## 8. Routers and report

In [ ]:
%%bash
set -e
$BIN/train-router --run "$RUN" --tau 0.8
$BIN/eval-routing --run "$RUN"

## 9. Share the results

The run folder on Drive holds `router_train.parquet`, `calib.parquet`, `test.parquet`,
`report.json` and `curves.csv` (a few MB). To hand them to Claude, upload them to a private
Hugging Face dataset: add a write token as a Colab secret named `HF_TOKEN` and set the repo id.

In [ ]:
HF_REPO = ""  # e.g. "your-username/custom-llm-router-results"
if HF_REPO:
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=RUN, repo_id=HF_REPO, repo_type="dataset", path_in_repo="t4",
                      ignore_patterns=["*.parts/*"])
    print(f"uploaded to https://huggingface.co/datasets/{HF_REPO}")